## One-Time Installation

In [1]:
!pip install evaluate
!pip install rouge_score
!pip install ragas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f382c488a297d5b43ffd1fd0e35f7af4a0f2a202b68be5c69acbb1e2022b01ab
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Imports & One-Time Initialisation

In [2]:
import pandas as pd
import ast
import asyncio
from typing import List

import evaluate as ev
from datasets import Dataset

# RAGAS
from ragas.metrics.collections import (
    AnswerRelevancy,
    Faithfulness,
    ContextPrecision,
    ContextRecall
)
from ragas.metrics import summarization_score
from ragas.llms.base import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas import evaluate

import openai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-1617001219.py:16: DeprecationWarning: Importing summarization_score from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import summarization_score
  from ragas.metrics import summarization_score


In [3]:
# Load OPENAI API KEY
import os
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# MODEL_NAME = "gpt-5.2-2025-12-11"

## Initialise Shared Models

In [6]:
def initialise_ragas_dependencies():
    client = openai.AsyncOpenAI()

    llm = llm_factory(
        "gpt-4o-mini",
        client=client,
        max_tokens=2048,
        temperature=0.0
    )

    embeddings = embedding_factory(
        "openai",
        model="text-embedding-3-large",
        client=client
    )

    metrics = {
        "answer_relevancy": AnswerRelevancy(
            llm=llm,
            embeddings=embeddings,
            strictness=3
        ),
        "faithfulness": Faithfulness(llm=llm),
        "context_precision": ContextPrecision(llm=llm),
        "context_recall": ContextRecall(llm=llm)
    }

    return metrics


## Utility: Context Normalisation

In [7]:
def parse_context(context):
    if isinstance(context, list):
        return context
    try:
        return ast.literal_eval(context)
    except Exception:
        return [context]


## BLEU & ROUGE (Per-Sample)

In [8]:
bleu = ev.load("bleu")
rouge = ev.load("rouge")

def compute_bleu_rouge(reference: str, prediction: str):
    bleu_score = bleu.compute(
        predictions=[prediction],
        references=[[reference]]
    )["bleu"]

    rouge_score = rouge.compute(
        predictions=[prediction],
        references=[reference]
    )["rougeL"]

    return bleu_score, rouge_score

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Summarisation Score (RAGAS)

In [9]:
def compute_summarization_score(contexts, response, reference_contexts):
    dataset = Dataset.from_dict({
        "contexts": [contexts],
        "response": [response],
        "reference_contexts": [reference_contexts]
    })

    score = evaluate(dataset, metrics=[summarization_score])
    return score["summary_score"][0]

## Core Row-wise Evaluation Function

In [10]:
async def evaluate_single_row(row, metrics):
    question = row["question"]
    response = row["llm_response"]
    reference = row["ground_truth_answer"]

    retrieved_contexts = parse_context(row["context_used"])
    supporting_contexts = parse_context(row["supporting_context"])

    # BLEU / ROUGE
    bleu_score, rougeL_score = compute_bleu_rouge(reference, response)

    # RAGAS metrics
    answer_relevancy = await metrics["answer_relevancy"].ascore(
        user_input=question,
        response=response
    )

    faithfulness = await metrics["faithfulness"].ascore(
        user_input=question,
        response=response,
        retrieved_contexts=retrieved_contexts
    )

    context_precision = await metrics["context_precision"].ascore(
        user_input=question,
        reference=reference,
        retrieved_contexts=retrieved_contexts
    )

    context_recall = await metrics["context_recall"].ascore(
        user_input=question,
        reference=reference,
        retrieved_contexts=retrieved_contexts
    )

    summarization = compute_summarization_score(
        contexts=retrieved_contexts,
        response=response,
        reference_contexts=supporting_contexts
    )

    return {
        "bleu": bleu_score,
        "rougeL": rougeL_score,
        "answer_relevancy": answer_relevancy.value,
        "faithfulness": faithfulness.value,
        "context_precision": context_precision.value,
        "context_recall": context_recall.value,
        "summarization_score": summarization
    }


## Full CSV Evaluation Pipeline

In [11]:
async def run_rag_evaluation_pipeline(
    input_csv: str,
    output_csv: str = "rag_evaluated_results.csv"
):
    df = pd.read_csv(input_csv)

    metrics = initialise_ragas_dependencies()

    results = []

    for _, row in df.iterrows():
        scores = await evaluate_single_row(row, metrics)
        results.append(scores)

    scores_df = pd.DataFrame(results)
    final_df = pd.concat([df.reset_index(drop=True), scores_df], axis=1)

    final_df.to_csv(output_csv, index=False)

    return final_df

## Running the Pipeline

In [12]:
# import asyncio
# import nest_asyncio

# nest_asyncio.apply()

# asyncio.run(
#     run_rag_evaluation_pipeline(
#         input_csv="/content/drive/MyDrive/MS-LJMU/Batch-Output/gpt-4o-mini-gt-llm-response.csv",
#         output_csv="/content/drive/MyDrive/MS-LJMU/Evaluation/result/gpt-4o-mini-gt-evaluation.csv"
#     )
# )

# print("RAG EValuation Completed Successfully!!")

In [13]:
final_df = await run_rag_evaluation_pipeline(
        input_csv="/content/drive/MyDrive/MS-LJMU/Batch-Output/llama-3.3-70b/llama-3.3-70b-gt-llm-response_5.csv",
        output_csv="/content/drive/MyDrive/MS-LJMU/Evaluation/result/llama-3.3-70b/llama-3.3-70b-gt-evaluation_5.csv"
    )

print("RAG EValuation Completed Successfully!!")

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

RAG EValuation Completed Successfully!!
